### Feature Importances

In [1]:
# Load Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load and Clean Data
df = pd.read_csv('data.csv')

# Clean percentage columns
df['Fair%'] = df['Fair%'].str.replace('%', '').astype(float) / 100
df['GIR%'] = df['GIR%'].str.replace('%', '').astype(float) / 100

# Create target variable: 1 if score over par (bogey or worse), else 0
df['Bogey_or_Worse'] = (df['+/-'] > 0).astype(int)

# Drop rows with missing data in key columns
df = df.dropna(subset=['Fair%', 'GIR%', 'Putts'])

# Feature Selection
features = ['Putts', 'Fair%', 'GIR%']
X = df[features]
y = df['Bogey_or_Worse']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# Feature Importance
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

print("\n🌲 Feature Importances:")
print(importance_df)


📊 Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         5

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


🌲 Feature Importances:
  Feature  Importance
2    GIR%    0.125630
1   Fair%    0.423419
0   Putts    0.450952


In [2]:
# === Step 1: Import Libraries ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import root_mean_squared_error


# === Step 2: Load Your Data ===
df = pd.read_csv('data.csv')  # Adjust path as needed

# === Step 3: Data Cleaning ===
df['Fair%'] = df['Fair%'].str.replace('%', '', regex=False).astype(float)
df['GIR%'] = df['GIR%'].str.replace('%', '', regex=False).astype(float)
df['Fairs'] = df['Fairs'].fillna(0)
df['Fair%'] = df['Fair%'].fillna(0)

# === Step 4: Feature Engineering ===
df['PuttsPerHole'] = df['Putts']
df['PuttsPerGIR'] = df.apply(lambda row: row['Putts'] if row['GIR'] == 1 else np.nan, axis=1)
df['ParType'] = df['Par'].apply(lambda x: f'Par {x}')

# Optional: fill missing PuttsPerGIR with average
df['PuttsPerGIR'] = df['PuttsPerGIR'].fillna(df['PuttsPerGIR'].mean())

# === Step 5: Prepare Model Inputs ===
features = ['Par', 'Putts', 'Fairs', 'Fair%', 'GIR', 'GIR%']
X = df[features]
y = df['TP']

# === Step 6: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Step 7: Train Model ===
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# === Step 8: Evaluate Model ===
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model RMSE: {rmse:.2f}")
print(f"Model R²: {r2:.2f}")

# === Step 9: Feature Importances ===
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature Importances:")
print(importances)

Model RMSE: 0.00
Model R²: 1.00

Feature Importances:
Par      0.0
Putts    0.0
Fairs    0.0
Fair%    0.0
GIR      0.0
GIR%     0.0
dtype: float64


### Shots Gained

In [3]:
import pandas as pd
import numpy as np

# 1. Load the Excel file
file_path = 'Golf Stats.xlsx'
hole_log = pd.read_excel(file_path, sheet_name='Hole Log')
course_stats = pd.read_excel(file_path, sheet_name='Course Stats')

# 2. Merge and Calculate Baseline
course_info = course_stats[['Course', 'Tees', 'Rating', 'Par']].rename(columns={'Par': 'Course_Par'})
df = hole_log.merge(course_info, on=['Course', 'Tees'], how='left')

# Expected Score for a Scratch Golfer (Baseline)
# If rating is missing, default to 72.0
df['Expected_Score'] = df['Par'] + (df['Rating'].fillna(72.0) - df['Course_Par'].fillna(72)) / 18

# 3. Calculate SG Components
df['SG_Total'] = df['Expected_Score'] - df['Hole Score']
df['is_gir'] = df['GIR'].astype(str).str.strip() == '1'
df['is_fwy'] = df['Fairways'].astype(str).str.strip() == '1'

# SG: Putting (Categorical Baseline)
df['Expected_Putts'] = np.where(df['is_gir'], 1.82, 1.40)
df['SG_Putting'] = df['Expected_Putts'] - df['Putts']
df['SG_T2G'] = df['SG_Total'] - df['SG_Putting']

# 4. Detailed Breakdown (OTT, APP, ARG)
df['SG_OTT'] = 0.0
df['SG_Approach'] = 0.0
df['SG_ARG'] = 0.0

# SG: Off-the-Tee (Par 4/5 only)
mask_p45 = df['Par'] > 3
df.loc[mask_p45, 'SG_OTT'] = np.where(df.loc[mask_p45, 'is_fwy'], 0.15, -0.15)

# SG: Approach
# Par 3s: All T2G is Approach
mask_p3 = df['Par'] == 3
df.loc[mask_p3, 'SG_Approach'] = df.loc[mask_p3, 'SG_T2G']
# Par 4/5: Fixed indexing
df.loc[mask_p45, 'SG_Approach'] = np.where(df.loc[mask_p45, 'is_gir'], 0.3, -0.2)

# SG: Around-the-Green (The Remainder)
df['SG_ARG'] = df['SG_T2G'] - df['SG_OTT'] - df['SG_Approach']

# 5. Create Round Summary
round_summary = df.groupby(['Golfer', 'Date', 'Course', 'Tees']).agg({
    'Hole Score': 'sum',
    'SG_Total': 'sum',
    'SG_T2G': 'sum',
    'SG_OTT': 'sum',
    'SG_Approach': 'sum',
    'SG_ARG': 'sum',
    'SG_Putting': 'sum'
}).reset_index()

# 6. Export to one file
with pd.ExcelWriter('strokes_gained_summary.xlsx') as writer:
    round_summary.to_excel(writer, sheet_name='Round Summary', index=False)
    df.to_excel(writer, sheet_name='Hole Detail', index=False)

print("File 'strokes_gained_summary.xlsx' has been created successfully!")

File 'strokes_gained_summary.xlsx' has been created successfully!


### Round Profiling

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# === Load Merged Data ===
xls = pd.ExcelFile("Golf Stats.xlsx")
putt_log = xls.parse("Putt Log")
sg_summary = pd.read_excel("strokes_gained_summary.xlsx")  # should include Golfer

# === Round-Level Stats (carry Golfer) ===
round_cols = [c for c in ['Golfer','Date','Course','Putts2','GIR','Fairways'] if c in putt_log.columns]
round_stats = putt_log[round_cols].copy()
round_stats.columns = [c if c != 'Putts2' else 'TotalPutts' for c in round_stats.columns]

# Clean dates
round_stats['Date'] = pd.to_datetime(round_stats['Date'], errors='coerce')
round_stats = round_stats.dropna(subset=['Date'])
sg_summary['Date'] = pd.to_datetime(sg_summary['Date'], errors='coerce')

# === Merge and Feature Engineering (keyed by Golfer, Date, Course) ===
merge_keys = [k for k in ['Golfer','Date','Course'] if k in sg_summary.columns and k in round_stats.columns]
df_all = pd.merge(sg_summary, round_stats, on=merge_keys, how='inner')

df_all['PuttsPerHole'] = df_all['TotalPutts'].astype(float) / 18
df_all['GIR%']        = df_all['GIR'].astype(float)        / 18
df_all['Fairway%']    = df_all['Fairways'].astype(float)   / 14

feature_cols = ['SG_OTT','SG_Approach','SG_Putting','GIR%','Fairway%','PuttsPerHole']

# === Per-golfer label maps ===
# Edit these to your preferred names per cluster ID for each golfer.
# If a golfer isn't listed here, we'll use cluster_labels_default.
cluster_labels_default = {
    0: "Off Day",
    1: "Fairway Putter",
    2: "Ball-Striker",
    3: "Greenskeeper",
    4: "Off Day"
}

cluster_labels_by_golfer = {
    "Tyler": {
        0: "All Average",
        1: "Ball Striker",
        2: "Greenskeeper",
        3: "Fairway Putter",
        4: "Ball Striker"
    },
    "Rich": {
        0: "Greenskeeper",
        1: "Ball Striker",
        2: "Off Day",
        3: "All Average",
        4: "Ball Striker"
    },
    "Ryan": {
        0: "All Average",
        1: "Off Day",
        2: "Off Day",
        3: "Greenskeeper",
        4: "Ball Striker"
    },
}

# === Holder for all golfers ===
all_out = []

for golfer, gdf in df_all.groupby('Golfer', sort=False):
    gdf = gdf.copy()

    # Select rows with complete features
    feats = gdf[feature_cols]
    feats_clean = feats.dropna()
    if feats_clean.empty:
        continue

    # Keep same rows
    gdf = gdf.loc[feats_clean.index].reset_index(drop=True)

    # Scale within golfer
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feats_clean)

    # Choose k safely (cannot exceed number of rows)
    k = min(5, len(gdf))
    if k < 2:
        gdf['Cluster'] = 0
    else:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        gdf['Cluster'] = kmeans.fit_predict(X_scaled)

    # Optional: inspect per-golfer cluster means
    cluster_means = gdf.groupby('Cluster')[feature_cols].mean()
    print(f"\n{golfer} — Cluster Averages:\n{cluster_means.round(3)}")

    # === Map labels using golfer-specific dict (fallback to default) ===
    labels = cluster_labels_by_golfer.get(golfer, cluster_labels_default)
    gdf['RoundType'] = gdf['Cluster'].map(labels).fillna('Cluster ' + gdf['Cluster'].astype(str))

    # Clean tiny float residuals and round
    tol = 1e-8
    for c in feature_cols:
        gdf[c] = gdf[c].apply(lambda x: 0.0 if isinstance(x,(int,float)) and abs(x) < tol else x)
    gdf[feature_cols] = gdf[feature_cols].round(3)

    all_out.append(gdf)

# === Save and Done ===
if all_out:
    out = pd.concat(all_out, ignore_index=True)

    # nice column order
    cols = [c for c in ['Golfer','Date','Course','TotalPutts','GIR','Fairways',
                        'PuttsPerHole','GIR%','Fairway%',
                        'SG_OTT','SG_Approach','SG_Putting',
                        'Cluster','RoundType'] if c in out.columns]
    out = out[cols]

    out.to_excel("clustered_rounds.xlsx", index=False)
    print("Clustered round profiles saved to 'clustered_rounds.xlsx'")
else:
    print("No rows with complete features to cluster.")



Rich — Cluster Averages:
         SG_OTT  SG_Approach  SG_Putting   GIR%  Fairway%  PuttsPerHole
Cluster                                                                
0        -0.488       -5.403      -5.275  0.069     0.393         1.722
1        -0.100       -1.593     -10.053  0.333     0.571         2.130
2        -0.937       -2.259      -5.925  0.347     0.286         1.875
3        -1.200       -3.580      -9.888  0.200     0.214         2.033
4        -1.800       -5.277      -8.170  0.083     0.071         1.889

Ryan — Cluster Averages:
         SG_OTT  SG_Approach  SG_Putting   GIR%  Fairway%  PuttsPerHole
Cluster                                                                
0          0.00       -2.253       -9.12  0.222     0.500         2.000
1         -0.75       -9.088      -15.75  0.139     0.286         2.333
2         -1.95       -9.517       -9.38  0.056     0.000         1.944
3         -2.10       -3.373      -18.96  0.111     0.000         2.500
4         -0

c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


### Clutch Score

In [5]:
import pandas as pd
from scipy.stats import linregress
import numpy as np

# === Load Hole Log Data ===
file_path = "Golf Stats.xlsx"
xls = pd.ExcelFile(file_path)
hole_log = xls.parse("Hole Log")

# === Clean and Prepare ===
# Ensure needed columns exist
required = ["Golfer", "Date", "Course", "Hole", "Score vs Par", "Handicap"]
missing = [c for c in required if c not in hole_log.columns]
if missing:
    raise ValueError(f"Missing required columns in 'Hole Log': {missing}")

# Normalize types
hole_log["Date"] = pd.to_datetime(hole_log["Date"], errors="coerce")
hole_log["Hole"] = pd.to_numeric(hole_log["Hole"], errors="coerce")
hole_log["Score vs Par"] = pd.to_numeric(hole_log["Score vs Par"], errors="coerce")
hole_log["Handicap"] = pd.to_numeric(hole_log["Handicap"], errors="coerce")

# Drop unusable rows
hole_log = hole_log.dropna(subset=["Golfer", "Date", "Course", "Hole", "Score vs Par", "Handicap"])

# === Filter to complete 18-hole rounds PER GOLFER ===
# Count holes by (Golfer, Date, Course)
hole_counts = hole_log.groupby(["Golfer", "Date", "Course"]).size()
valid_rounds = hole_counts[hole_counts == 18].index

# Keep only complete rounds
hole_log_filtered = (
    hole_log
    .set_index(["Golfer", "Date", "Course"])
    .loc[valid_rounds]
    .reset_index()
)

# === Clutch Score Function (operates on a single golfer's single round) ===
def compute_clutch_metrics(group: pd.DataFrame) -> pd.Series:
    g = group.sort_values("Hole")

    early = g[g["Hole"] <= 15]
    final = g[g["Hole"] >= 16]

    # Unweighted clutch (higher = better)
    avg_early = early["Score vs Par"].mean()
    sum_final = final["Score vs Par"].sum()
    clutch_score_par = (avg_early * 3) - sum_final

    # Difficulty weighting by handicap (lower handicap = harder hole → slightly higher weight)
    early_w = early.assign(Weight=1 + (18 - early["Handicap"]) / 36)
    final_w = final.assign(Weight=1 + (18 - final["Handicap"]) / 36)

    early_weighted = (early_w["Score vs Par"] * early_w["Weight"]).mean()
    final_weighted = (final_w["Score vs Par"] * final_w["Weight"]).sum()
    clutch_score_weighted = (early_weighted * 3) - final_weighted

    # Trend over 18 holes (slope of score vs hole number)
    slope = linregress(g["Hole"], g["Score vs Par"]).slope

    return pd.Series(
        {
            "Avg_Early_Score_vs_Par": avg_early,
            "Final3_Score_vs_Par": sum_final,
            "ClutchScore_Par": clutch_score_par,
            "ClutchScore_Weighted": clutch_score_weighted,
            "Score_Trend_Slope": slope,
        }
    )

# === Compute Clutch Scores Per (Golfer, Date, Course) ===
clutch_scores = (
    hole_log_filtered
    .groupby(["Golfer", "Date", "Course"], group_keys=False)
    .apply(compute_clutch_metrics)
    .reset_index()
)

# === Labeling Function (same thresholds for all golfers; tweak if you want per-golfer rules) ===
def label_clutch(score: float) -> str:
    if score >= 1.5:
        return "Strong Finish"
    elif score >= 0.5:
        return "Solid Finish"
    elif score >= -0.5:
        return "Neutral"
    else:
        return "Drop-off"

clutch_scores["ClutchLabel"] = clutch_scores["ClutchScore_Weighted"].apply(label_clutch)

# Clean tiny float residuals and round
TOL = 1e-8
num_cols = [
    "Avg_Early_Score_vs_Par",
    "Final3_Score_vs_Par",
    "ClutchScore_Par",
    "ClutchScore_Weighted",
    "Score_Trend_Slope",
]
for c in num_cols:
    clutch_scores[c] = clutch_scores[c].apply(lambda x: 0.0 if isinstance(x, (int, float)) and abs(x) < TOL else x)
clutch_scores[num_cols] = clutch_scores[num_cols].round(3)

# === Export to Excel ===
clutch_scores.to_excel("clutch_score_analysis.xlsx", index=False)
print("✅ Clutch score analysis complete. Results saved to 'clutch_score_analysis.xlsx'")

✅ Clutch score analysis complete. Results saved to 'clutch_score_analysis.xlsx'


C:\Users\Owner\AppData\Local\Temp\ipykernel_39720\1000279394.py:76: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_clutch_metrics)


### Club Recommendations

In [6]:
import pandas as pd
import numpy as np

# === Load Range Stats Data ===
xls = pd.ExcelFile("Golf Stats.xlsx")
range_stats_df = xls.parse("Range Stats")

# === Clean Columns ===
range_stats_df['Total Distance'] = pd.to_numeric(range_stats_df['Total Distance'], errors='coerce')
range_stats_df['Club'] = range_stats_df['Club Type'].str.lower().str.strip()

# === Filter Valid Shots ===
filtered_df = range_stats_df[
    (range_stats_df['Total Distance'] > 100) &
    (range_stats_df['Total Distance'] <= 225) &
    (range_stats_df['Club'] != '1i')
].copy()

# === Bin Total Distances ===
bin_edges = np.arange(100, 225, 5)
bin_labels = [f"{int(b)}-{int(b+5)}" for b in bin_edges[:-1]]
filtered_df['DistanceBin'] = pd.cut(filtered_df['Total Distance'], bins=bin_edges, labels=bin_labels, include_lowest=True)

# === Aggregate by Club and Bin: use Q1 (top 25%) ===
def q1(series):
    return series.quantile(0.50)  # Top 25% threshold (not bottom)(.75)

club_summary = filtered_df.groupby(['DistanceBin', 'Club'], observed=True).agg(
    Top25Carry=('Total Distance', q1),
    ShotCount=('Total Distance', 'count')
).reset_index()

best_clubs = (
    club_summary
    .sort_values(by=['DistanceBin', 'Top25Carry', 'ShotCount'], ascending=[True, False, False])
    .groupby('DistanceBin', observed=True)
    .first()
    .reset_index()
)
best_clubs.to_excel("club_selector.xlsx", index=False)
print("✅ Club recommendations saved to 'club_selector.xlsx'")

✅ Club recommendations saved to 'club_selector.xlsx'


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# === Step 1: Load Data ===
xls = pd.ExcelFile("Golf Stats.xlsx")
df = xls.parse("Range Stats")

# === Step 2: Clean and Prepare Data ===
df['Total Distance'] = pd.to_numeric(df['Total Distance'], errors='coerce')
df['Club'] = df['Club Type'].str.lower().str.strip()

# Normalize variants and exclude invalid clubs
df['Club'] = df['Club'].replace({
    'average': np.nan
})
df = df[df['Club'].notna() & (df['Club'] != '1i')]

# Select and clean ML features
features = ['Ball Speed', 'Launch Angle', 'Club Speed', 'Smash Factor']
df[features] = df[features].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=features + ['Total Distance'])

# === Step 3: Train Model ===
X = df[features]
y = df['Total Distance']
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# === Step 4: Predict Total Distance and Build Club Ranges ===
# Irons: 35th–85th (5i/6i upper = 90th). Wedges etc.: 25th–90th. 3h/3w: upper = 95th. Driver: median–max.
IRON_CLUBS = ('pw', '9i', '8i', '7i', '6i', '5i')
df['PredictedTotal'] = model.predict(X)

grp = df.groupby('Club')['PredictedTotal']
p25 = grp.quantile(0.25)
p90 = grp.quantile(0.90)
p35 = grp.quantile(0.35)
p85 = grp.quantile(0.85)
p95 = grp.quantile(0.95)
p50 = grp.quantile(0.50)
pmax = grp.max()  # 100th percentile

club_ranges = pd.concat([p25, p90], axis=1).reset_index()
club_ranges.columns = ['Club', 'Lower', 'Upper']

mask_iron = club_ranges['Club'].isin(IRON_CLUBS)
club_ranges.loc[mask_iron, 'Lower'] = club_ranges.loc[mask_iron, 'Club'].map(p35)
club_ranges.loc[mask_iron, 'Upper'] = club_ranges.loc[mask_iron, 'Club'].map(p85)

mask_56 = club_ranges['Club'].isin(['5i', '6i'])
club_ranges.loc[mask_56, 'Upper'] = club_ranges.loc[mask_56, 'Club'].map(p90)

mask_fw = club_ranges['Club'].isin(['3h', '3w'])
club_ranges.loc[mask_fw, 'Upper'] = club_ranges.loc[mask_fw, 'Club'].map(p95)

mask_d = club_ranges['Club'] == 'd'
club_ranges.loc[mask_d, 'Lower'] = club_ranges.loc[mask_d, 'Club'].map(p50)
club_ranges.loc[mask_d, 'Upper'] = club_ranges.loc[mask_d, 'Club'].map(pmax)

club_ranges['Range'] = club_ranges.apply(lambda row: f"{int(row['Lower'])}-{int(row['Upper'])}", axis=1)

# Filter to valid clubs
valid_clubs = ['lw', 'sw', 'gw', 'pw', '9i', '8i', '7i', '6i', '5i', '3h', '3w', 'd']
club_ranges = club_ranges[club_ranges['Club'].isin(valid_clubs)].sort_values('Lower').reset_index(drop=True)

# === Step 5: Save to Excel ===
club_ranges.to_excel("club_ranges.xlsx", index=False)
print("✅ Refined club total distance ranges saved to 'club_ranges.xlsx'")

# === Step 6: Distance-to-Club Lookup ===
def recommend_club(distance, club_data):
    for _, row in club_data.iterrows():
        if row['Lower'] <= distance <= row['Upper']:
            return row['Club'], row['Range']
    return None, None

# === Step 7: Use the recommender ===
user_input = input("Enter a total distance (e.g. 155) or a club name (e.g. 8i): ").strip().lower()

if user_input.isdigit():
    target = int(user_input)
    club, club_range = recommend_club(target, club_ranges)
    if club:
        print(f"✅ Recommended Club: {club.title()} (Range: {club_range})")
    else:
        print("⚠️ No club found with a consistent total distance range for that number.")

elif user_input in club_ranges['Club'].values:
    row = club_ranges[club_ranges['Club'] == user_input].iloc[0]
    band_notes = {'d': 'median–max', '3h': '25th–95th', '3w': '25th–95th', **{c: ('35th–90th' if c in ('5i', '6i') else '35th–85th') for c in IRON_CLUBS}}
    band = band_notes.get(row['Club'], '25th–90th')
    print(f"📊 Club: {row['Club'].title()} | Range: {row['Range']} ({band} of predicted yards)")
else:
    print("⚠️ Invalid input. Please enter a number or valid club name.")


✅ Refined club total distance ranges saved to 'club_ranges.xlsx'
✅ Recommended Club: Gw (Range: 102-124)


### Golf Recommendations

In [8]:
import pandas as pd
import numpy as np
import re

# === Load Club Ranges ===
club_ranges = pd.read_excel("club_ranges.xlsx")
club_ranges['Club'] = club_ranges['Club'].str.lower().str.strip()

# === Load Hole Log ===
xls = pd.ExcelFile("Golf Stats.xlsx")
df = xls.parse("Hole Log")
df.columns = df.columns.str.strip()

# === Clean columns ===
df['Fairways'] = pd.to_numeric(df['Fairways'], errors='coerce').fillna(0)
df['Putts'] = pd.to_numeric(df['Putts'], errors='coerce').fillna(0)
df['GIR'] = pd.to_numeric(df['GIR'], errors='coerce').fillna(0)
df['Yardage'] = pd.to_numeric(df['Yardage'], errors='coerce')
df['Par'] = pd.to_numeric(df['Par'], errors='coerce')
df['Fair%'] = df['Fairways']
df['GIR%'] = df['GIR']

# === Strategy Recommendation (Only by Course and Hole) ===
# === Strategy Recommendation: Use AVERAGE stats per Course + Hole
strategy_input = df.groupby(['Course', 'Hole']).agg(
    AvgFair=('Fairways', 'mean'),
    AvgGIR=('GIR', 'mean'),
    AvgPutts=('Putts', 'mean'),
    Par=('Par', 'mean')
).reset_index()

def recommend_strategy_avg(row):
    detailed = []
    short = []

    if row['Par'] != 3 and row['AvgFair'] <= 0.25:
        detailed.append("Work on Tee Shot (Change Strategy)")
        short.append("Focus on Tee Shot")

    if row['AvgGIR'] < 0.2:
        if row['Par'] == 3:
            detailed.append("Focus on Tee Shot (Change Strategy)")
            short.append("Tee Shot")
        else:
            detailed.append("Focus on Approach (Change Strategy)")
            short.append("Approach Shot")

    if row['AvgPutts'] >= 3:
        detailed.append("Focus on Putting (Try Lag Putting)")
        short.append("Putting")

    if not detailed:
        return "No Issue"
    elif len(detailed) == 1:
        return detailed[0]
    else:
        return "; ".join(short)

# Apply the corrected strategy function
strategy_input['Recommendation'] = strategy_input.apply(recommend_strategy_avg, axis=1)

# Final strategy_map
strategy_map = strategy_input[['Course', 'Hole', 'Recommendation']]

# === Utility Functions ===
def get_avg(club_name):
    row = club_ranges[club_ranges['Club'] == club_name]
    if not row.empty:
        return (row.iloc[0]['Lower'] + row.iloc[0]['Upper']) / 2
    return np.nan

driver_avg = get_avg('d') or 235
threew_avg = get_avg('3w') or 215

def recommend_club(distance, club_data):
    club_data = club_data[club_data['Club'] != 'd']
    for _, row in club_data.iterrows():
        if row['Lower'] <= distance <= row['Upper']:
            return row['Club'].title()
    layup_subtract = 50 if distance <= 250 else 100
    layup_target = distance - layup_subtract
    for _, row in club_data.iterrows():
        if row['Lower'] <= layup_target <= row['Upper']:
            return f"Layup ({row['Club'].title()})"
    return "Layup"

def is_bad_followup(club):
    club = str(club).lower()
    return club.startswith("layup") or club in ["3w", "3h"]

def format_club_recommendation(text):
    match = re.match(r'Layup \((.*?)\)', str(text), re.IGNORECASE)
    if match:
        club = match.group(1).upper()
        return f"Layup ({club})"
    return str(text).upper()

# === Compute Tee & Approach Recommendations ===
def club_recommendation(row):
    if pd.isna(row['Yardage']) or pd.isna(row['Par']):
        return pd.Series(["N/A", "", np.nan])

    yardage = float(row['Yardage'])
    par = int(row['Par'])

    if par == 3:
        target = yardage
        tee_club = recommend_club(target, club_ranges)
        return pd.Series([tee_club, "", round(target)])

    d_target = yardage - driver_avg
    w3_target = yardage - threew_avg

    d_result = recommend_club(d_target, club_ranges)
    w3_result = recommend_club(w3_target, club_ranges)

    d_bad = is_bad_followup(d_result)
    w3_bad = is_bad_followup(w3_result)

    if not w3_bad and d_bad:
        tee_club = "3W"
        target = w3_target
        approach_club = w3_result
    else:
        tee_club = "D"
        target = d_target
        approach_club = d_result

    return pd.Series([tee_club, approach_club, round(target)])

df[['Tee Recommendation', 'Approach Recommendation', 'Target Distance']] = df.apply(club_recommendation, axis=1)

# === Summarize by Course + Hole + Tees for Club Recommendations ===
summary_df = df.groupby(['Course', 'Hole', 'Tees'], as_index=False).agg(
    Yardage=('Yardage', 'mean'),
    Par=('Par', 'mean'),
    Target_Distance=('Target Distance', 'mean'),
    Total_Putts=('Putts', 'sum'),
    Total_Fairways=('Fairways', 'sum'),
    Total_GIR=('GIR', 'sum'),
    Attempts=('Hole', 'count'),
    Tee_Recommendation=('Tee Recommendation', lambda x: x.mode().iloc[0] if not x.mode().empty else "N/A"),
    Approach_Recommendation=('Approach Recommendation', lambda x: x.mode().iloc[0] if not x.mode().empty else "")
)

# Merge back the strategy recommendation (not split by tee)
summary_df = pd.merge(summary_df, strategy_map, on=['Course', 'Hole'], how='left')

# === Convert raw counts to percentages ===
summary_df['Putts %'] = (summary_df['Total_Putts'] / summary_df['Attempts']).round(2)
summary_df['Fairways %'] = (summary_df['Total_Fairways'] / summary_df['Attempts']).round(2)
summary_df['GIR %'] = (summary_df['Total_GIR'] / summary_df['Attempts']).round(2)

# === Final Formatting ===
summary_df = summary_df[[
    'Course', 'Hole', 'Tees', 'Yardage', 'Par', 'Target_Distance',
    'Putts %', 'Fairways %', 'GIR %', 'Recommendation',
    'Tee_Recommendation', 'Approach_Recommendation'
]]

summary_df.rename(columns={
    'Target_Distance': 'Target Distance',
    'Putts %': 'Putts',
    'Fairways %': 'Fairways',
    'GIR %': 'GIR',
    'Tee_Recommendation': 'Tee Recommendation',
    'Approach_Recommendation': 'Approach Recommendation'
}, inplace=True)

summary_df['Approach Recommendation'] = summary_df['Approach Recommendation'].apply(format_club_recommendation)
summary_df['Tee Recommendation'] = summary_df['Tee Recommendation'].apply(lambda x: x.upper())

# === Output ===
from IPython.display import display
print("\n📊 Summary by Course, Hole, and Tees (Strategy NOT split by Tees):")
display(summary_df)

summary_df.to_excel("golf_recommendations.xlsx", index=False)
print("✅ Summary saved to 'golf_recommendations.xlsx'")



📊 Summary by Course, Hole, and Tees (Strategy NOT split by Tees):


,Course,Hole,Tees,Yardage,Par,Target Distance,Putts,Fairways,GIR,Recommendation,Tee Recommendation,Approach Recommendation
0,Bethpage Black,1,White,429.0,4.0,186.0,2.0,0.0,0.0,Focus on Tee Shot; Approach Shot,D,5I
1,Bethpage Black,2,White,354.0,4.0,111.0,2.0,0.0,0.0,Focus on Tee Shot; Approach Shot,D,GW
2,Bethpage Black,3,White,158.0,3.0,158.0,2.0,0.0,0.0,Focus on Tee Shot (Change Strategy),8I,
3,Bethpage Black,4,White,461.0,5.0,218.0,1.0,1.0,0.0,Focus on Approach (Change Strategy),D,3W
4,Bethpage Black,5,White,423.0,4.0,180.0,2.0,0.0,0.0,Focus on Tee Shot; Approach Shot,D,6I
...,...,...,...,...,...,...,...,...,...,...,...,...
373,Wind Watch Hamlet Course,14,Blue,218.0,3.0,218.0,2.0,0.0,0.0,Focus on Tee Shot (Change Strategy),3W,
374,Wind Watch Hamlet Course,15,Blue,543.0,5.0,300.0,2.0,1.0,0.0,Focus on Approach (Change Strategy),D,Layup (3H)
375,Wind Watch Hamlet Course,16,Blue,316.0,4.0,73.0,2.0,0.0,0.0,Focus on Tee Shot; Approach Shot,D,LW
376,Wind Watch Hamlet Course,17,Blue,419.0,4.0,176.0,2.0,0.0,0.0,Focus on Tee Shot; Approach Shot,D,6I


✅ Summary saved to 'golf_recommendations.xlsx'
